## Data Preparation

In [ ]:
import pandas as pd

df = pd.read_csv('data/cbtz.csv')

df

In [ ]:
# total rows
len(df)

In [ ]:
# keep only distinct rows
df = df.drop_duplicates(subset=['ChillBuddy Response'])

In [ ]:
# find distinct rows
len(df), df['ChillBuddy Response'].nunique()

In [ ]:
# write to CSV
df.to_csv('data/cbtz_distinct.csv', index=False)

In [ ]:
system_prompt = "You are ChillBuddy: A Gen Z AI pal for mental wellness. Use empathy, validation, CBT (esp. cognitive reframing & small steps). Keep it real, supportive, & chat-friendly. ✨"

In [ ]:
# Format the data into JSONL for OpenAI fine-tuning
import json

def format_for_openai_tuning(row, system_prompt):
    """Format a single row into the OpenAI fine-tuning JSONL format"""
    # Create the messages array with system, user, and assistant messages
    messages = [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": row["User Input"]},
        {"role": "assistant", "content": row["ChillBuddy Response"]}
    ]
    
    # Create the final JSON object
    json_obj = {"messages": messages}
    
    # Return the JSON object as a string
    return json.dumps(json_obj, ensure_ascii=False)

# Split into 80:10:10 for train, val, and test
train = df.sample(frac=0.8, random_state=42)
val = df.drop(train.index).sample(frac=0.5, random_state=42)
test = df.drop(train.index).drop(val.index)

# Apply the function to each row in the dataframe
jsonl_train_data = train.apply(lambda row: format_for_openai_tuning(row, system_prompt), axis=1)
jsonl_val_data = val.apply(lambda row: format_for_openai_tuning(row, system_prompt), axis=1)
jsonl_test_data = test.apply(lambda row: format_for_openai_tuning(row, system_prompt), axis=1)

# Save the JSONL data to a file
with open('data/cbtz_train.jsonl', 'w', encoding='utf-8') as f:
    for json_str in jsonl_train_data:
        f.write(json_str + '\n')

with open('data/cbtz_val.jsonl', 'w', encoding='utf-8') as f:
    for json_str in jsonl_val_data:
        f.write(json_str + '\n')
        
with open('data/cbtz_test.jsonl', 'w', encoding='utf-8') as f:
    for json_str in jsonl_test_data:
        f.write(json_str + '\n')

print(f"JSONL file created")

## Fine-Tuning GPT-4o-mini Model

In this section, we'll fine-tune the gpt-4o-mini-2024-07-18 model using our prepared data and integrate with Weights & Biases (wandb) for experiment tracking.


In [18]:
import os
import time
import openai
import wandb
from datetime import datetime

# Set OpenAI API key (make sure to set this in your environment or replace with your key)
# openai.api_key = os.environ.get('OPENAI_API_KEY')

# Initialize OpenAI client
client = openai.OpenAI()

In [ ]:
def fine_tune_model(base_model, training_file, validation_file, n_epochs, batch_size, learning_rate_multiplier):
    """
    Fine-tune an OpenAI model with specified hyperparameters and track with wandb.
    
    Args:
        base_model (str): The base model to fine-tune
        training_file (str): The ID of the training file
        validation_file (str): The ID of the validation file
        n_epochs (int): Number of epochs for training
        batch_size (int): Batch size for training
        learning_rate_multiplier (float): Learning rate multiplier
        
    Returns:
        str: The ID of the fine-tuned model
    """
    
    # Create a unique run name with the cbtz- prefix
    run_name = f"cbtz-{base_model.split('/')[-1]}-e{n_epochs}-b{batch_size}-lr{learning_rate_multiplier}"
    
    # Initialize wandb
    # wandb.init(
    #     project="nlp-ft",
    #     name=run_name,
    #     config={
    #         "base_model": base_model,
    #         "n_epochs": n_epochs,
    #         "batch_size": batch_size,
    #         "learning_rate_multiplier": learning_rate_multiplier,
    #         "training_file": training_file,
    #         "validation_file": validation_file
    #     }
    # )
    
    print(f"Starting fine-tuning job with name: {run_name}")
    
    # Create fine-tuning job
    job = client.fine_tuning.jobs.create(
        training_file=training_file,
        validation_file=validation_file,
        model=base_model,
        hyperparameters={
            "n_epochs": n_epochs,
            "batch_size": batch_size,
            "learning_rate_multiplier": learning_rate_multiplier
        },
        integrations=[{
            "type": "wandb",
            "wandb": {
                "project": "nlp-ft",
                "name": run_name,
            }
        }],
        suffix=run_name  # This will be added to the model name
    )
    
    job_id = job.id
    print(f"Fine-tuning job created: {job_id}")
    
    # Track job status
    status = job.status
    # wandb.log({"job_status": status})
    
    # Poll for job status until it's complete
    while status not in ["succeeded", "failed", "cancelled"]:
        print(f"Job status: {status}. Waiting...")
        time.sleep(60)  # Check every minute
        
        # Get the latest job status
        job = client.fine_tuning.jobs.retrieve(job_id)
        new_status = job.status
        
        # Log status change to wandb
        if new_status != status:
            status = new_status
            # wandb.log({"job_status": status})
            
            # If we have training metrics, log them to wandb
            # if hasattr(job, 'training_metrics') and job.training_metrics:
            #     metrics = job.training_metrics
            #     wandb.log({
            #         "training_loss": metrics.get('training_loss', 0),
            #         "training_token_accuracy": metrics.get('training_token_accuracy', 0)
            #     })
            
            # If we have validation metrics, log them to wandb
            # if hasattr(job, 'validation_metrics') and job.validation_metrics:
            #     metrics = job.validation_metrics
            #     wandb.log({
            #         "validation_loss": metrics.get('validation_loss', 0),
            #         "validation_token_accuracy": metrics.get('validation_token_accuracy', 0)
            #     })
    
    # Get final job details
    final_job = client.fine_tuning.jobs.retrieve(job_id)
    
    # Log final status
    print(f"Fine-tuning job completed with status: {final_job.status}")
    # wandb.log({"final_status": final_job.status})
    
    # Log final metrics if available
    # if hasattr(final_job, 'training_metrics') and final_job.training_metrics:
        # wandb.log({"final_training_loss": final_job.training_metrics.get('training_loss', 0)})
    
    # if hasattr(final_job, 'validation_metrics') and final_job.validation_metrics:
        # wandb.log({"final_validation_loss": final_job.validation_metrics.get('validation_loss', 0)})
    
    # Get the fine-tuned model ID
    fine_tuned_model = final_job.fine_tuned_model
    
    if fine_tuned_model:
        print(f"Fine-tuned model ID: {fine_tuned_model}")
        # wandb.log({"fine_tuned_model_id": fine_tuned_model})
    else:
        print("No fine-tuned model was created.")
    
    # Finish wandb run
    # wandb.finish()
    
    return fine_tuned_model


In [ ]:
# Define hyperparameter combinations to explore
hyperparameter_configs = [
    {"n_epochs": 1, "batch_size": 4, "learning_rate_multiplier": 0.2},
    {"n_epochs": 2, "batch_size": 4, "learning_rate_multiplier": 1.0},
    {"n_epochs": 2, "batch_size": 8, "learning_rate_multiplier": 0.2},
    {"n_epochs": 3, "batch_size": 8, "learning_rate_multiplier": 1.0},
    {"n_epochs": 3, "batch_size": 16, "learning_rate_multiplier": 0.2}
]

# Set the base model and file IDs
base_model = "gpt-4o-mini-2024-07-18"
training_file = "file-P2d8n3J8Zjp59cXBa2w31f"
validation_file = "file-4bdgmr6BjHykwZuAB6r5A9"

# Dictionary to store fine-tuned model IDs
fine_tuned_models = {}

# Run fine-tuning for each hyperparameter configuration
for config in hyperparameter_configs:
    print(f"Fine-tuning with config: {config}")
    
    # Create a key for this configuration
    config_key = f"e{config['n_epochs']}_b{config['batch_size']}_lr{config['learning_rate_multiplier']}"
    
    # Run fine-tuning
    fine_tuned_model = fine_tune_model(
        base_model=base_model,
        training_file=training_file,
        validation_file=validation_file,
        n_epochs=config['n_epochs'],
        batch_size=config['batch_size'],
        learning_rate_multiplier=config['learning_rate_multiplier']
    )
    
    # Store the fine-tuned model ID
    fine_tuned_models[config_key] = fine_tuned_model
    
    print(f"Completed fine-tuning for config: {config}")

# Print summary of all fine-tuned models
print("Summary of fine-tuned models:")
for config_key, model_id in fine_tuned_models.items():
    print(f"Config {config_key}: {model_id}")


## Model Evaluation

In this section, we'll evaluate the fine-tuned models using multiple metrics:
1. Perplexity (log-probabilities)
2. Text-overlap metrics (BLEU, ROUGE, edit distance)
3. Embedding-similarity methods


In [20]:
# Install required packages for evaluation
# !pip install nltk rouge-score Levenshtein sentence-transformers

# Import necessary libraries
import nltk
import numpy as np
import pandas as pd
from nltk.translate.bleu_score import sentence_bleu, SmoothingFunction
from rouge_score import rouge_scorer
import Levenshtein
from sentence_transformers import SentenceTransformer
from scipy.spatial.distance import cosine
import json
import math

# Download necessary NLTK data
nltk.download('punkt')
nltk.download('punkt_tab')

[nltk_data] Downloading package punkt to /Users/sila/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to /Users/sila/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


True

In [21]:
# Load test data
def load_test_data(file_path):
    test_data = []
    with open(file_path, 'r', encoding='utf-8') as f:
        for line in f:
            example = json.loads(line)
            test_data.append(example)
    return test_data

# Load test data
test_data = load_test_data('data/cbtz_test.jsonl')
print(f"Loaded {len(test_data)} test examples")

# Extract system prompts, user inputs, and expected outputs
system_prompts = []
user_inputs = []
expected_outputs = []

for example in test_data:
    messages = example['messages']
    system_prompts.append(messages[0]['content'])  # System message
    user_inputs.append(messages[1]['content'])     # User message
    expected_outputs.append(messages[2]['content']) # Assistant message

print(f"Sample system prompt: {system_prompts[0][:50]}...")
print(f"Sample user input: {user_inputs[0][:50]}...")
print(f"Sample expected output: {expected_outputs[0][:50]}...")


Loaded 10 test examples
Sample system prompt: You are ChillBuddy: A Gen Z AI pal for mental well...
Sample user input: After feeling unstoppable, I’m now hopeless and ca...
Sample expected output: Whoa, that swing from unstoppable to hopeless & st...


In [22]:
# Function to generate responses from a model
def generate_responses(model_id, system_prompts, user_inputs, max_samples=None):
    """Generate responses from a specified model for evaluation."""
    responses = []
    log_probs = []
    
    # Limit the number of samples if specified
    if max_samples is not None and max_samples < len(user_inputs):
        indices = np.random.choice(len(user_inputs), max_samples, replace=False)
        system_prompts = [system_prompts[i] for i in indices]
        user_inputs = [user_inputs[i] for i in indices]
    
    print(f"Generating responses from model {model_id} for {len(user_inputs)} examples...")
    
    for i, (system_prompt, user_input) in enumerate(zip(system_prompts, user_inputs)):
        try:
            # Create messages for the API call
            messages = [
                {"role": "system", "content": system_prompt},
                {"role": "user", "content": user_input}
            ]
            
            # Generate response with log probabilities
            response = client.chat.completions.create(
                model=model_id,
                messages=messages,
                temperature=0,  # Use deterministic output for evaluation
                logprobs=True,
                top_logprobs=1
            )
            
            # Extract response text
            response_text = response.choices[0].message.content
            responses.append(response_text)
            
            # Extract log probabilities if available
            if hasattr(response.choices[0], 'logprobs') and response.choices[0].logprobs:
                # Calculate average log probability
                token_logprobs = [lp.logprob for lp in response.choices[0].logprobs.content]
                avg_logprob = sum(token_logprobs) / len(token_logprobs) if token_logprobs else None
                log_probs.append(avg_logprob)
            else:
                log_probs.append(None)
            
            # Print progress
            if (i + 1) % 5 == 0 or i == len(user_inputs) - 1:
                print(f"Generated {i + 1}/{len(user_inputs)} responses")
                
        except Exception as e:
            print(f"Error generating response for example {i}: {e}")
            responses.append(None)
            log_probs.append(None)
    
    return responses, log_probs


In [23]:
# Evaluation metrics functions

# 1. Perplexity calculation
def calculate_perplexity(log_probs):
    """Calculate perplexity from log probabilities."""
    valid_log_probs = [lp for lp in log_probs if lp is not None]
    if not valid_log_probs:
        return None
    
    # Perplexity is exp(-average log probability)
    avg_log_prob = sum(valid_log_probs) / len(valid_log_probs)
    perplexity = math.exp(-avg_log_prob)
    return perplexity

# 2. BLEU score calculation
def calculate_bleu(generated_responses, expected_outputs):
    """Calculate BLEU scores for generated responses."""
    bleu_scores = []
    smoothing = SmoothingFunction().method1
    
    for gen, exp in zip(generated_responses, expected_outputs):
        if gen is None:
            bleu_scores.append(None)
            continue
            
        # Tokenize the sentences
        gen_tokens = nltk.word_tokenize(gen.lower())
        exp_tokens = nltk.word_tokenize(exp.lower())
        
        # Calculate BLEU score
        try:
            bleu = sentence_bleu([exp_tokens], gen_tokens, smoothing_function=smoothing)
            bleu_scores.append(bleu)
        except Exception as e:
            print(f"Error calculating BLEU score: {e}")
            bleu_scores.append(None)
    
    # Calculate average BLEU score
    valid_scores = [s for s in bleu_scores if s is not None]
    avg_bleu = sum(valid_scores) / len(valid_scores) if valid_scores else None
    
    return bleu_scores, avg_bleu

# 3. ROUGE score calculation
def calculate_rouge(generated_responses, expected_outputs):
    """Calculate ROUGE scores for generated responses."""
    scorer = rouge_scorer.RougeScorer(['rouge1', 'rouge2', 'rougeL'], use_stemmer=True)
    rouge_scores = []
    
    for gen, exp in zip(generated_responses, expected_outputs):
        if gen is None:
            rouge_scores.append(None)
            continue
            
        # Calculate ROUGE scores
        try:
            scores = scorer.score(exp, gen)
            rouge_scores.append(scores)
        except Exception as e:
            print(f"Error calculating ROUGE score: {e}")
            rouge_scores.append(None)
    
    # Calculate average ROUGE scores
    valid_scores = [s for s in rouge_scores if s is not None]
    
    if not valid_scores:
        return rouge_scores, None, None, None
    
    avg_rouge1 = sum(s['rouge1'].fmeasure for s in valid_scores) / len(valid_scores)
    avg_rouge2 = sum(s['rouge2'].fmeasure for s in valid_scores) / len(valid_scores)
    avg_rougeL = sum(s['rougeL'].fmeasure for s in valid_scores) / len(valid_scores)
    
    return rouge_scores, avg_rouge1, avg_rouge2, avg_rougeL

# 4. Edit distance calculation
def calculate_edit_distance(generated_responses, expected_outputs):
    """Calculate normalized edit distances for generated responses."""
    edit_distances = []
    
    for gen, exp in zip(generated_responses, expected_outputs):
        if gen is None:
            edit_distances.append(None)
            continue
            
        # Calculate edit distance
        try:
            distance = Levenshtein.distance(gen, exp)
            # Normalize by the length of the longer string
            norm_distance = distance / max(len(gen), len(exp))
            edit_distances.append(norm_distance)
        except Exception as e:
            print(f"Error calculating edit distance: {e}")
            edit_distances.append(None)
    
    # Calculate average edit distance
    valid_distances = [d for d in edit_distances if d is not None]
    avg_distance = sum(valid_distances) / len(valid_distances) if valid_distances else None
    
    return edit_distances, avg_distance

# 5. Embedding similarity calculation
def calculate_embedding_similarity(generated_responses, expected_outputs):
    """Calculate embedding similarities for generated responses."""
    # Load sentence transformer model
    model = SentenceTransformer('all-MiniLM-L6-v2')
    
    similarities = []
    
    for gen, exp in zip(generated_responses, expected_outputs):
        if gen is None:
            similarities.append(None)
            continue
            
        # Calculate embeddings
        try:
            gen_embedding = model.encode(gen)
            exp_embedding = model.encode(exp)
            
            # Calculate cosine similarity (1 - cosine distance)
            similarity = 1 - cosine(gen_embedding, exp_embedding)
            similarities.append(similarity)
        except Exception as e:
            print(f"Error calculating embedding similarity: {e}")
            similarities.append(None)
    
    # Calculate average similarity
    valid_similarities = [s for s in similarities if s is not None]
    avg_similarity = sum(valid_similarities) / len(valid_similarities) if valid_similarities else None
    
    return similarities, avg_similarity


In [24]:
# Function to evaluate a model
def evaluate_model(model_id, system_prompts, user_inputs, expected_outputs, max_samples=10):
    """Evaluate a model using multiple metrics."""
    print(f"Evaluating model: {model_id}")
    
    # Initialize wandb run for evaluation
    wandb.init(
        project="nlp-ft",
        name=f"eval-{model_id.split('/')[-1]}",
        config={
            "model_id": model_id,
            "num_samples": min(max_samples, len(user_inputs))
        }
    )
    
    # Generate responses
    generated_responses, log_probs = generate_responses(
        model_id, system_prompts, user_inputs, max_samples
    )
    
    # Limit expected outputs to match generated responses
    if max_samples is not None and max_samples < len(expected_outputs):
        indices = np.random.choice(len(expected_outputs), max_samples, replace=False)
        expected_outputs = [expected_outputs[i] for i in indices]
    
    # Calculate metrics
    
    # 1. Perplexity
    perplexity = calculate_perplexity(log_probs)
    print(f"Perplexity: {perplexity:.4f}")
    wandb.log({"perplexity": perplexity})
    
    # 2. BLEU score
    _, avg_bleu = calculate_bleu(generated_responses, expected_outputs)
    print(f"Average BLEU score: {avg_bleu:.4f}")
    wandb.log({"bleu_score": avg_bleu})
    
    # 3. ROUGE scores
    _, avg_rouge1, avg_rouge2, avg_rougeL = calculate_rouge(generated_responses, expected_outputs)
    print(f"Average ROUGE-1: {avg_rouge1:.4f}")
    print(f"Average ROUGE-2: {avg_rouge2:.4f}")
    print(f"Average ROUGE-L: {avg_rougeL:.4f}")
    wandb.log({
        "rouge1": avg_rouge1,
        "rouge2": avg_rouge2,
        "rougeL": avg_rougeL
    })
    
    # 4. Edit distance
    _, avg_edit_distance = calculate_edit_distance(generated_responses, expected_outputs)
    print(f"Average normalized edit distance: {avg_edit_distance:.4f}")
    wandb.log({"edit_distance": avg_edit_distance})
    
    # 5. Embedding similarity
    _, avg_embedding_similarity = calculate_embedding_similarity(generated_responses, expected_outputs)
    print(f"Average embedding similarity: {avg_embedding_similarity:.4f}")
    wandb.log({"embedding_similarity": avg_embedding_similarity})
    
    # Log sample responses
    sample_indices = np.random.choice(len(generated_responses), min(3, len(generated_responses)), replace=False)
    for i in sample_indices:
        if generated_responses[i] is not None:
            wandb.log({
                f"sample_{i}_user": user_inputs[i],
                f"sample_{i}_expected": expected_outputs[i],
                f"sample_{i}_generated": generated_responses[i]
            })
    
    # Finish wandb run
    wandb.finish()
    
    # Return evaluation results
    return {
        "perplexity": perplexity,
        "bleu_score": avg_bleu,
        "rouge1": avg_rouge1,
        "rouge2": avg_rouge2,
        "rougeL": avg_rougeL,
        "edit_distance": avg_edit_distance,
        "embedding_similarity": avg_embedding_similarity
    }


In [27]:
"""
fine_tuned_models = {'e1_b4_lr0.2': 'ft:gpt-4o-mini-2024-07-18:personal:cbtz-gpt-4o-mini-2024-07-18-e1-b4-lr0-2:BPgLfvKq',
 'e2_b4_lr1.0': 'ft:gpt-4o-mini-2024-07-18:personal:cbtz-gpt-4o-mini-2024-07-18-e2-b4-lr1-0:BPgSZ6oU',
 'e2_b8_lr0.2': 'ft:gpt-4o-mini-2024-07-18:personal:cbtz-gpt-4o-mini-2024-07-18-e2-b8-lr0-2:BPgYkCJf',
 'e3_b8_lr1.0': 'ft:gpt-4o-mini-2024-07-18:personal:cbtz-gpt-4o-mini-2024-07-18-e3-b8-lr1-0:BPgfgRpv',
 'e3_b16_lr0.2': 'ft:gpt-4o-mini-2024-07-18:personal:cbtz-gpt-4o-mini-2024-07-18-e3-b16-lr0-2:BPgm9QkP'}
"""
base_model = "gpt-4o-mini-2024-07-18"
fine_tuned_models = {
    "e1_b4_lr0.2": "ft:gpt-4o-mini-2024-07-18:personal:cbtz-gpt-4o-mini-2024-07-18-e1-b4-lr0-2:BPgLfvKq",
    "e2_b4_lr1.0": "ft:gpt-4o-mini-2024-07-18:personal:cbtz-gpt-4o-mini-2024-07-18-e2-b4-lr1-0:BPgSZ6oU",
    "e2_b8_lr0.2": "ft:gpt-4o-mini-2024-07-18:personal:cbtz-gpt-4o-mini-2024-07-18-e2-b8-lr0-2:BPgYkCJf",
    "e3_b8_lr1.0": "ft:gpt-4o-mini-2024-07-18:personal:cbtz-gpt-4o-mini-2024-07-18-e3-b8-lr1-0:BPgfgRpv",
    "e3_b16_lr0.2": "ft:gpt-4o-mini-2024-07-18:personal:cbtz-gpt-4o-mini-2024-07-18-e3-b16-lr0-2:BPgm9QkP",
}

In [28]:
# Evaluate all fine-tuned models
evaluation_results = {}

# Add the base model for comparison
models_to_evaluate = {"base_model": base_model}
models_to_evaluate.update(fine_tuned_models)

for model_name, model_id in models_to_evaluate.items():
    print(f"\nEvaluating {model_name}: {model_id}")
    
    # Evaluate the model
    results = evaluate_model(
        model_id,
        system_prompts,
        user_inputs,
        expected_outputs,
        max_samples=10  # Limit to 10 samples for faster evaluation
    )
    
    # Store the results
    evaluation_results[model_name] = results

# Create a comparison dataframe
comparison_df = pd.DataFrame(evaluation_results).T

# Display the comparison
print("\nModel Comparison:")
display(comparison_df)

# Save the comparison to CSV
comparison_df.to_csv('model_evaluation_results.csv')
print("Evaluation results saved to model_evaluation_results.csv")



Evaluating base_model: gpt-4o-mini-2024-07-18
Evaluating model: gpt-4o-mini-2024-07-18


Generating responses from model gpt-4o-mini-2024-07-18 for 10 examples...
Generated 5/10 responses
Generated 10/10 responses
Perplexity: 1.3621
Average BLEU score: 0.0150
Average ROUGE-1: 0.2717
Average ROUGE-2: 0.0714
Average ROUGE-L: 0.1660
Average normalized edit distance: 0.7624
Average embedding similarity: 0.6328


bleu_score,▁
edit_distance,▁
embedding_similarity,▁
perplexity,▁
rouge1,▁
rouge2,▁
rougeL,▁
bleu_score,0.015
edit_distance,0.76238
embedding_similarity,0.63276
perplexity,1.3621



Evaluating e1_b4_lr0.2: ft:gpt-4o-mini-2024-07-18:personal:cbtz-gpt-4o-mini-2024-07-18-e1-b4-lr0-2:BPgLfvKq
Evaluating model: ft:gpt-4o-mini-2024-07-18:personal:cbtz-gpt-4o-mini-2024-07-18-e1-b4-lr0-2:BPgLfvKq


Generating responses from model ft:gpt-4o-mini-2024-07-18:personal:cbtz-gpt-4o-mini-2024-07-18-e1-b4-lr0-2:BPgLfvKq for 10 examples...
Generated 5/10 responses
Generated 10/10 responses
Perplexity: 1.4137
Average BLEU score: 0.0161
Average ROUGE-1: 0.3083
Average ROUGE-2: 0.0790
Average ROUGE-L: 0.1796
Average normalized edit distance: 0.7346
Average embedding similarity: 0.6402


...

Generating responses from model ft:gpt-4o-mini-2024-07-18:personal:cbtz-gpt-4o-mini-2024-07-18-e3-b16-lr0-2:BPgm9QkP for 10 examples...
Generated 5/10 responses
Generated 10/10 responses
Perplexity: 1.3880
Average BLEU score: 0.0156
Average ROUGE-1: 0.3109
Average ROUGE-2: 0.0809
Average ROUGE-L: 0.1872
Average normalized edit distance: 0.7388
Average embedding similarity: 0.6510


bleu_score,▁
edit_distance,▁
embedding_similarity,▁
perplexity,▁
rouge1,▁
rouge2,▁
rougeL,▁
bleu_score,0.01557
edit_distance,0.73882
embedding_similarity,0.65101
perplexity,1.38801



Model Comparison:


,perplexity,bleu_score,rouge1,rouge2,rougeL,edit_distance,embedding_similarity
base_model,1.362105,0.014996,0.271692,0.071410,0.165962,0.762375,0.632759
e1_b4_lr0.2,1.413677,0.016120,0.308331,0.079006,0.179554,0.734607,0.640220
e2_b4_lr1.0,2.054872,0.114315,0.417170,0.184115,0.328065,0.658376,0.770158
e2_b8_lr0.2,1.445246,0.015453,0.301280,0.073182,0.181785,0.737255,0.670549
e3_b8_lr1.0,2.196107,0.098762,0.407744,0.172542,0.327930,0.654532,0.782578
e3_b16_lr0.2,1.388010,0.015568,0.310899,0.080862,0.187240,0.738818,0.651011


Evaluation results saved to model_evaluation_results.csv
